In [0]:
# ==========================================
# 04. INGESTA BRONZE - SUCURSALES
# Proyecto: ElectroCasa
# Método de ingesta: COPY INTO
# ==========================================

RUTA_SUCURSALES = "/Volumes/proyecto_final/default/vol_landing/sucursales/sucursales.csv"

df_sucursales = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RUTA_SUCURSALES)
)

print("ESQUEMA DEL ARCHIVO:")
df_sucursales.printSchema()

print("TOTAL DE REGISTROS:", df_sucursales.count())

display(df_sucursales.limit(10))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS proyecto_final.bronze.sucursales (
    sucursal_id STRING,
    nombre_sucursal STRING,
    distrito STRING,
    ciudad STRING,
    region STRING,
    fecha_apertura DATE,
    archivo_origen STRING,
    fecha_ingesta TIMESTAMP
)
USING DELTA;

In [0]:
%sql
COPY INTO proyecto_final.bronze.sucursales
FROM (
    SELECT
        sucursal_id,
        nombre_sucursal,
        distrito,
        ciudad,
        region,
        TO_DATE(fecha_apertura, 'yyyy-MM-dd') AS fecha_apertura,
        _metadata.file_name AS archivo_origen,
        current_timestamp() AS fecha_ingesta
    FROM '/Volumes/proyecto_final/default/vol_landing/sucursales'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
);

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.sucursales
GROUP BY archivo_origen
ORDER BY archivo_origen;